In [ ]:
%pip install ipykernel
%pip install opencv-python 
%pip install matplotlib
%pip install matplotlib tk
%pip install plotly nbformat
%pip install scipy
%pip install pyqtgraph PyOpenGL PyQt5
%matplotlib qt


In [ ]:
import time
from collections import deque
from dataclasses import dataclass

import cv2 as cv
import matplotlib
import numpy as np

In [1]:
import threading
import time
from collections import deque
from dataclasses import dataclass

import cv2 as cv
import matplotlib
import numpy as np


def use_window_backend():
    """Notebooks default to an inline backend that closes live figures."""
    try:
        shell = get_ipython()
    except NameError:
        return
    for backend in ("qt", "tk"):
        try:
            shell.run_line_magic("matplotlib", backend)
            return
        except Exception:
            continue


use_window_backend()
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.cm import ScalarMappable  # noqa: E402
from matplotlib.colors import Normalize  # noqa: E402


FRAME_SHAPE = (384, 256, 2)
FRAME_SIZE = int(np.prod(FRAME_SHAPE))
FONT = cv.FONT_HERSHEY_SIMPLEX


@dataclass
class Config:
    camera_index: int = 0
    left_crop: int = 98              # gray (fallback) mode only
    zoom: int = 4
    rows: int = 80                   # time rows kept in the 3D views
    columns: int = 64                # target number of 3D columns across the image
    update_seconds: float = 0.08     # time between 3D updates
    figure_size: tuple = (18, 8)
    blur_x: float = 0.8              # smoothing across image columns (3D views)
    blur_t: float = 1.0              # smoothing across time rows (3D views)

    # temperature scale (raw mode). Starts at temp_min/max, then follows the scene.
    temp_min: float = 10.0
    temp_max: float = 50.0
    auto_scale: bool = True
    scale_low_pct: float = 1.0
    scale_high_pct: float = 99.8
    min_span: float = 6.0            # never zoom the scale tighter than this many degrees
    scale_smoothing: float = 0.08    # 0..1, lower = steadier scale

    # wisps
    ambient_pct: float = 30.0        # "normal" = this percentile of the recent history
    contrast: float = 0.15           # fraction of the scale span that makes a wisp fully strong
    min_strength: float = 0.15       # weaker cells are not drawn


class Scale:
    """Colour/height range shared by the camera view, the bar and both 3D views."""

    def __init__(self, low, high, unit, auto, cfg):
        self.low, self.high, self.unit, self.auto, self.cfg = low, high, unit, auto, cfg
        self.primed = False

    @property
    def span(self):
        return max(self.high - self.low, 1e-6)

    @property
    def label(self):
        return "°C" if self.unit == "C" else self.unit

    def update(self, values):
        """Follow the scene smoothly; rise fast so hot things never clip."""
        if not self.auto:
            return
        cfg = self.cfg
        lo, hi = np.percentile(values[::2, ::2], (cfg.scale_low_pct, cfg.scale_high_pct))
        hi = max(hi, lo + cfg.min_span)
        if not self.primed:
            self.low, self.high, self.primed = float(lo), float(hi), True
            return
        self.low += cfg.scale_smoothing * (lo - self.low)
        self.high += (0.5 if hi > self.high else cfg.scale_smoothing) * (hi - self.high)


def decode_temps(frame):
    """Return a 192x256 array in °C from a raw TC001 frame."""
    data = frame.reshape(FRAME_SHAPE)[FRAME_SHAPE[0] // 2:]
    raw = data[..., 0].astype(np.uint16) + data[..., 1].astype(np.uint16) * 256
    return (raw / 64.0 - 273.15).astype(np.float32)


def decode_gray(frame, crop):
    """Return the cropped brightness image (0 to 255) from a normal colour frame."""
    return cv.cvtColor(frame[:, crop:], cv.COLOR_BGR2GRAY).astype(np.float32)


def open_raw(index):
    """Return a capture that delivers raw TC001 frames, or None if it doesn't."""
    for name, backend in (("DSHOW", cv.CAP_DSHOW), ("MSMF", cv.CAP_MSMF), ("default", cv.CAP_ANY)):
        capture = cv.VideoCapture(index, backend)
        if not capture.isOpened():
            capture.release()
            continue
        capture.set(cv.CAP_PROP_CONVERT_RGB, 0)
        ok, frame = capture.read()
        print(f"[{name}] read ok={ok}, shape={getattr(frame, 'shape', None)}, "
              f"size={getattr(frame, 'size', None)} (need {FRAME_SIZE})")
        if ok and frame.size == FRAME_SIZE:
            return capture
        capture.release()
    return None


class Camera:
    """Reads frames on a background thread so the display never works from stale, queued frames."""

    def __init__(self, cfg):
        self.crop = cfg.left_crop
        self.capture = open_raw(cfg.camera_index)
        self.raw = self.capture is not None
        if not self.raw:
            self.capture = cv.VideoCapture(cfg.camera_index)
        if not self.capture.isOpened():
            raise RuntimeError("Could not open camera. Try another camera_index.")
        if self.raw:
            self.scale = Scale(cfg.temp_min, cfg.temp_max, "C", cfg.auto_scale, cfg)
        else:
            self.scale = Scale(0, 255, "signal", False, cfg)

        self._frame = None
        self._lock = threading.Lock()
        self._ready = threading.Event()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._grab, daemon=True)
        self._thread.start()

    def _grab(self):
        while not self._stop.is_set():
            ok, frame = self.capture.read()
            with self._lock:
                self._frame = frame if ok else None
            self._ready.set()
            if not ok:
                break

    def read(self):
        """Return the newest temperatures in °C (raw mode) or brightness, or None."""
        if not self._ready.wait(2.0):
            return None
        self._ready.clear()
        with self._lock:
            frame = self._frame
        if frame is None:
            return None
        return decode_temps(frame) if self.raw else decode_gray(frame, self.crop)

    def release(self):
        self._stop.set()
        self._thread.join(timeout=2.0)
        self.capture.release()


class CameraView:
    """Camera window with a hotspot marker and a temperature bar."""

    def __init__(self, cfg, scale):
        self.zoom = cfg.zoom
        self.scale = scale
        self.bar = None

    def make_bar(self, height):
        gradient = np.linspace(255, 0, height).astype(np.uint8)[:, None].repeat(22, axis=1)
        return cv.applyColorMap(gradient, cv.COLORMAP_INFERNO)

    def show(self, values):
        s = self.scale
        scaled = np.clip((values - s.low) / s.span, 0, 1)
        image = cv.applyColorMap(np.uint8(scaled * 255), cv.COLORMAP_INFERNO)
        image = cv.resize(image, None, fx=self.zoom, fy=self.zoom, interpolation=cv.INTER_CUBIC)
        h, w = image.shape[:2]

        y, x = np.unravel_index(np.argmax(values), values.shape)
        cx, cy = int((x + 0.5) * self.zoom), int((y + 0.5) * self.zoom)
        cv.drawMarker(image, (cx, cy), (255, 255, 255), cv.MARKER_CROSS, 14, 1)
        cv.putText(image, f"{values[y, x]:.1f} {s.unit}", (min(cx + 8, w - 120), max(cy - 8, 16)),
                   FONT, 0.5, (255, 255, 255), 1, cv.LINE_AA)

        if self.bar is None or self.bar.shape[0] != h:
            self.bar = self.make_bar(h)
        labels = np.zeros((h, 80, 3), np.uint8)
        for i in range(6):
            frac = i / 5
            row = int(frac * (h - 1))
            cv.line(labels, (0, row), (5, row), (255, 255, 255), 1)
            cv.putText(labels, f"{s.high - frac * s.span:.1f}", (9, min(max(row + 5, 12), h - 3)),
                       FONT, 0.45, (255, 255, 255), 1, cv.LINE_AA)
        spacer = np.zeros((h, 8, 3), np.uint8)
        cv.imshow("TC001", np.hstack([image, spacer, self.bar, labels]))

    @staticmethod
    def quit_requested():
        return cv.waitKey(1) & 0xFF in (ord("q"), 27)


class History:
    """Rolling rows: hottest value in each block of image columns, averaged over each time step."""

    def __init__(self, width, cfg):
        self.step = max(1, width // cfg.columns)
        self.n = width // self.step
        self.x = np.arange(self.n) * self.step + self.step / 2
        self.rows = deque(maxlen=cfg.rows)
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def add_frame(self, values):
        blurred = cv.GaussianBlur(values, (0, 0), 1.0)        # suppress single hot pixels
        h = blurred.shape[0]
        blocks = blurred[:, :self.n * self.step].reshape(h, self.n, self.step)
        self.total += blocks.max(axis=(0, 2))                  # hottest pixel per column block
        self.count += 1

    def commit(self):
        if self.count == 0:
            return
        row = self.total / self.count
        if not self.rows:
            self.rows.extend([row] * self.rows.maxlen)
        else:
            self.rows.append(row)
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def as_array(self):
        return np.array(self.rows, np.float32)


def style_dark_axes(ax, xlim, ylim, zlabel):
    ax.set_facecolor("black")
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((0, 0, 0, 0))
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Image X", color="0.7")
    ax.set_ylabel("Time", color="0.7")
    ax.set_zlabel(zlabel, color="0.7")
    ax.tick_params(colors="0.6")
    ax.view_init(45, -70)


class SurfacePanel:
    """Left panel: surface of the hottest value per column over time, in real units."""

    def __init__(self, ax, history, cfg, scale):
        self.ax, self.scale, self.surf = ax, scale, None
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))
        style_dark_axes(ax, (0, history.x.max()), (0, cfg.rows - 1), f"Signal ({scale.label})")
        ax.set_title("3D map", color="0.85")

    def draw(self, grid):
        s = self.scale
        if self.surf is not None:
            self.surf.remove()
        self.surf = self.ax.plot_surface(
            self.x, self.y, np.clip(grid, s.low, s.high),
            cmap="inferno", vmin=s.low, vmax=s.high,
            linewidth=0, antialiased=False, shade=False,
            rcount=grid.shape[0], ccount=grid.shape[1],
        )
        self.ax.set_zlim(s.low, s.high)


class WispPanel:
    """Right panel: particles for everything hotter than the scene's normal level."""

    PARTICLES = 2

    def __init__(self, ax, history, cfg, scale):
        self.ax, self.cfg, self.scale, self.history = ax, cfg, scale, history
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))
        rng = np.random.default_rng(7)
        self.jitter = rng.uniform(-0.5, 0.5, (self.PARTICLES, 3, self.x.size))   # fixed, so no flicker
        self.fade = np.linspace(0.15, 1.0, cfg.rows)[:, None]
        self.ambient = None
        self.top = None
        style_dark_axes(ax, (0, history.x.max()), (0, cfg.rows - 1), f"{scale.label} above normal")
        ax.set_title("Wisps", color="0.85")
        self.cloud = ax.scatter([0], [0], [0], s=1, c=[(0, 0, 0, 0)], depthshade=False, linewidths=0)

    def draw(self, grid):
        s, cfg = self.scale, self.cfg
        target = float(np.percentile(grid, cfg.ambient_pct))
        self.ambient = target if self.ambient is None else self.ambient + 0.05 * (target - self.ambient)

        contrast = max(cfg.contrast * s.span, 1e-6)
        excess = np.clip(grid - self.ambient, 0, None)
        strength = np.clip(excess / contrast, 0, 1) * self.fade
        cells = np.flatnonzero(strength.ravel() > cfg.min_strength)

        target_top = max(4 * contrast, float(excess.max()) * 1.1)
        self.top = target_top if self.top is None else self.top + 0.2 * (target_top - self.top)
        self.ax.set_zlim(0, self.top)

        if not cells.size:
            self.cloud._offsets3d = ([0.0], [0.0], [0.0])
            self.cloud.set_facecolor([(0, 0, 0, 0)])
            self.cloud.set_sizes([1])
            return

        heat = excess.ravel()[cells]
        power = strength.ravel()[cells]
        jit = self.jitter[:, :, cells]                                        # (P, 3, n)
        px = (self.x.ravel()[cells] + jit[:, 0] * self.history.step).ravel()
        py = (self.y.ravel()[cells] + jit[:, 1]).ravel()
        pz = np.clip(heat + jit[:, 2] * contrast * 0.15, 0, None).ravel()
        power = np.tile(power, self.PARTICLES)
        temp = self.ambient + np.tile(heat, self.PARTICLES)                    # absolute value -> same colour bar

        colors = plt.get_cmap("inferno")(np.clip((temp - s.low) / s.span, 0, 1))
        colors[:, 3] = np.clip(power * 1.2, 0, 1)
        self.cloud._offsets3d = (px, py, pz)
        self.cloud.set_facecolor(colors)
        self.cloud.set_sizes(4 + 34 * power)


class SideBySide:
    """One window: surface map (left), wisp map (right), shared temperature bar."""

    def __init__(self, history, cfg, scale):
        plt.ion()
        self.history, self.cfg, self.scale = history, cfg, scale
        self.fig = plt.figure("Thermal 3D + wisps", figsize=cfg.figure_size, facecolor="black")
        left = self.fig.add_subplot(121, projection="3d")
        right = self.fig.add_subplot(122, projection="3d")
        self.fig.subplots_adjust(left=0.01, right=0.9, bottom=0.02, top=0.95, wspace=0.02)

        self.mappable = ScalarMappable(norm=Normalize(scale.low, scale.high), cmap="inferno")
        self.cbar = self.fig.colorbar(self.mappable, cax=self.fig.add_axes([0.93, 0.2, 0.014, 0.6]))
        self.cbar.set_label(f"Temperature ({scale.label})", color="0.8")
        self.cbar.ax.tick_params(colors="0.8")
        self.cbar.outline.set_edgecolor("0.5")

        self.panels = [
            SurfacePanel(left, history, cfg, scale),
            WispPanel(right, history, cfg, scale),
        ]

    def draw(self):
        """Redraw both panels. Returns False once the window is closed."""
        grid = cv.GaussianBlur(self.history.as_array(), (0, 0),
                               sigmaX=self.cfg.blur_x, sigmaY=self.cfg.blur_t)
        self.mappable.set_clim(self.scale.low, self.scale.high)
        self.cbar.update_normal(self.mappable)
        for panel in self.panels:
            panel.draw(grid)
        plt.pause(0.001)
        return plt.fignum_exists(self.fig.number)


def run(cfg=Config()):
    if "inline" in matplotlib.get_backend().lower():
        raise RuntimeError("Inline backend closes the figure. Run %matplotlib qt, then restart the kernel.")

    camera = Camera(cfg)
    print("Reading temperatures (°C)." if camera.raw else "Reading brightness only: this camera sends processed frames.")
    view = CameraView(cfg, camera.scale)
    history = maps = None
    last_update = 0.0

    try:
        while True:
            values = camera.read()
            if values is None:
                break
            camera.scale.update(values)

            if history is None:
                history = History(values.shape[1], cfg)
                maps = SideBySide(history, cfg, camera.scale)

            view.show(values)
            history.add_frame(values)

            if time.monotonic() - last_update >= cfg.update_seconds:
                last_update = time.monotonic()
                history.commit()
                if not maps.draw():
                    break

            if view.quit_requested():
                break
    finally:
        camera.release()
        cv.destroyAllWindows()
        plt.close("all")


if __name__ == "__main__":
    run()

[DSHOW] read ok=True, shape=(512, 484, 3), size=743424 (need 196608)
[MSMF] read ok=True, shape=(1, 495616), size=495616 (need 196608)
[default] read ok=True, shape=(1, 495616), size=495616 (need 196608)
Reading brightness only: this camera sends processed frames.
